# Advance NLP Lab — Questions 2, 4 and 5

Simple Jupyter Notebook solutions, following the practical style of the Question 1 spaCy solution.

- **Q2:** Information retrieval-based Question Answering using TF-IDF and cosine similarity
- **Q4:** Topic modelling using Latent Dirichlet Allocation (LDA)
- **Q5:** WordNet synonyms, definitions, and context-based word-sense identification

## Setup

In [ ]:
%pip install -q scikit-learn nltk

# Question 2: Information Retrieval-based Question Answering

**Aim:** Accept a question and retrieve the most relevant sentence from a small document collection.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Natural Language Processing enables computers to understand human language.",
    "SpaCy is a Python library used for natural language processing and named entity recognition.",
    "TF-IDF represents text using word importance scores.",
    "Question answering systems retrieve or generate answers to user questions.",
    "Latent Dirichlet Allocation is a topic modelling algorithm.",
    "WordNet is a lexical database containing words, synonyms, and definitions."
]

# Split documents into sentences (each example is already one sentence).
sentences = documents

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(sentences)

def answer_question(question):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, document_vectors).flatten()
    best_index = scores.argmax()

    print("Question:", question)
    print("Retrieved answer:", sentences[best_index])
    print("Similarity score:", round(float(scores[best_index]), 3))

answer_question("What is WordNet used for?")

In [ ]:
# Try your own question
question = "Which library performs named entity recognition?"
answer_question(question)

# Question 4: Topic Modelling using LDA

**Aim:** Apply Latent Dirichlet Allocation to a small collection of text documents and display the top five words for each topic.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

documents = [
    "The team won the football match after scoring two goals.",
    "The player trained daily before the tournament and match.",
    "The doctor treated patients and prescribed medicine in the hospital.",
    "Nurses and doctors work in hospitals to improve patient health.",
    "Python is used for machine learning and artificial intelligence.",
    "Machine learning models learn patterns from data using algorithms."
]

# Convert text into a document-term matrix.
vectorizer = CountVectorizer(stop_words="english")
doc_term_matrix = vectorizer.fit_transform(documents)

# Discover 3 topics.
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(doc_term_matrix)

words = vectorizer.get_feature_names_out()

for topic_index, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-5:][::-1]
    top_words = [words[i] for i in top_indices]
    print(f"Topic {topic_index + 1}: {', '.join(top_words)}")

In [ ]:
# Show the strongest topic assignment for each document.
topic_scores = lda.transform(doc_term_matrix)

for i, scores in enumerate(topic_scores):
    print(f"Document {i + 1}: Topic {scores.argmax() + 1} | {documents[i]}")

# Question 5: WordNet Synonyms, Definitions and Context

**Aim:** Use NLTK WordNet to list synonyms and definitions for a word, and use the Lesk algorithm to estimate its meaning in a sentence.

In [ ]:
import nltk

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

In [ ]:
def explore_word(word):
    synsets = wn.synsets(word)

    if not synsets:
        print("No WordNet entries found for:", word)
        return

    print("Word:", word)
    print("\nPossible meanings:")

    for i, synset in enumerate(synsets, start=1):
        synonyms = sorted({
            lemma.name().replace("_", " ")
            for lemma in synset.lemmas()
        })
        print(f"\n{i}. Synset: {synset.name()}")
        print("   Definition:", synset.definition())
        print("   Synonyms:", ", ".join(synonyms))

explore_word("bank")

In [ ]:
# Context-based word-sense identification with the Lesk algorithm.
sentences = [
    "She deposited her money at the bank.",
    "They sat on the bank of the river."
]

for sentence in sentences:
    sense = lesk(sentence.split(), "bank")
    print("\nSentence:", sentence)

    if sense:
        print("Predicted meaning:", sense.definition())
        print("Synonyms:", ", ".join(
            lemma.name().replace("_", " ") for lemma in sense.lemmas()
        ))
    else:
        print("No sense could be selected.")

## Summary

| Question | Method | Output |
|---|---|---|
| Q2 | TF-IDF + cosine similarity | Most relevant sentence for a question |
| Q4 | CountVectorizer + LDA | Top five words per topic and topic per document |
| Q5 | NLTK WordNet + Lesk | Synonyms, definitions, and context-based word sense |

**Note:** Q2 is a retrieval-based QA baseline: it returns an existing sentence rather than generating a new answer. LDA topics and Lesk word-sense results are approximate, especially on small datasets.